In [1]:
"""
    Connect to the workspace
    Reads subscription / resource group / workspace from config.json in this folder,
    and authenticates with the account you signed in to via `az login`.
"""
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print(ml_client.workspace_name)

Found the config file in: .\config.json


ws-azureml-practice


In [2]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input
from azure.ai.ml import automl

In [3]:
my_training_data_input = Input(
    type=AssetTypes.MLTABLE,
    path="azureml:input-data-automl:1")

In [4]:
classification_job = automl.classification(
    compute="aml-cluster",
    experiment_name="auto-ml-class-dev",
    training_data=my_training_data_input,
    target_column_name="Diabetic",
    primary_metric="accuracy",
    n_cross_validations=5,
    enable_model_explainability=True
)

In [5]:
"""
    Specify the primary metric
    AutoML uses primary metric to rank all trained models and select the best one. AML supports a set of named
    metrics for each task type.
"""
from azure.ai.ml.automl import ClassificationPrimaryMetrics
list(ClassificationPrimaryMetrics)

[<ClassificationPrimaryMetrics.AUC_WEIGHTED: 'AUCWeighted'>,
 <ClassificationPrimaryMetrics.ACCURACY: 'Accuracy'>,
 <ClassificationPrimaryMetrics.NORM_MACRO_RECALL: 'NormMacroRecall'>,
 <ClassificationPrimaryMetrics.AVERAGE_PRECISION_SCORE_WEIGHTED: 'AveragePrecisionScoreWeighted'>,
 <ClassificationPrimaryMetrics.PRECISION_SCORE_WEIGHTED: 'PrecisionScoreWeighted'>]

In [6]:
"""
    Set the metric
    
    Each model AutoML trains consumes compute resources. To control costs and training time, you can set limits on
    an AutoML job using set_limits()
    
    Several options
    - timeout_limit: Number of mins after which the complete AutoML experiment is terminated
    - trial_timeout_limit: Number of mins after which a single trial can take
    - max_trials: Maximum number of trials, or models that are trained
    - enable_early_termination: Whether to end the experiment if the score isn's improving in the short term
"""
classification_job.set_limits(
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=5,
    enable_early_termination=True
)
"""
    To save time, you can also run multiple trials in parallel. When you use a compute cluster, 
    you can have as many parallel trails as you have nodes. The maximum number of parallel trials
    is therefore related to the maximum number of nodes your compute cluster has.
    
    If you want to set a limit on the number of parallel trials, use max_concurrent_trials.
"""

'\n    To save time, you can also run multiple trials in parallel. When you use a compute cluster, \n    you can have as many parallel trails as you have nodes. The maximum number of parallel trials\n    is therefore related to the maximum number of nodes your compute cluster has.\n\n    If you want to set a limit on the number of parallel trials, use max_concurrent_trials.\n'

In [9]:
"""
    Set the training properties
    
    AutoML tries various combinations of featurization and algos to train a ml model. If you already
    know that certain algos wont work, can include or exclude.
"""

'\n    Set the training properties\n\n    AutoML tries various combinations of featurization and algos to train a ml model. If you already\n    know that certain algos wont work, can include or exclude.\n'

In [9]:
"""
    Submit an AutoML experiment
"""
returned_job = ml_client.jobs.create_or_update(classification_job)

In [10]:
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

Monitor your job at https://ml.azure.com/runs/great_match_qpclhxcncm?wsid=/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourcegroups/rg-azureml-practice/workspaces/ws-azureml-practice&tid=723a5a87-f39a-4a22-9247-3fc240c01396


In [4]:
import mlflow
mlflow.set_tracking_uri(
    "azureml://canadacentral.api.azureml.ms/mlflow/v1.0/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice"
)
mlflow.set_experiment("local-test")
with mlflow.start_run():
    mlflow.log_metric("x", 1)

2026/10/08 19:38:51 INFO mlflow.tracking.fluent: Experiment with name 'local-test' does not exist. Creating a new experiment.


🏃 View run happy_van_mq8b22dj at: https://canadacentral.api.azureml.ms/mlflow/v2.0/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice/#/experiments/86eead44-8649-4312-a5d9-5f5dc5d5e44b/runs/872e6aa5-6b68-42a9-80f2-cc1627f6d780
🧪 View experiment at: https://canadacentral.api.azureml.ms/mlflow/v2.0/subscriptions/cb426598-2b99-4c4d-9e84-da661ea30aa2/resourceGroups/rg-azureml-practice/providers/Microsoft.MachineLearningServices/workspaces/ws-azureml-practice/#/experiments/86eead44-8649-4312-a5d9-5f5dc5d5e44b
